In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re

d:\Program\envs\agent_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# Đọc nội dung từ file markdown của bạn (ví dụ file từ trang 1301 - 1495)
def read_markdown_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

In [3]:
# 1. Tiền xử lý nhẹ: Đôi khi các tiêu đề trong markdown có chứa thẻ in đậm (VD: # **SPIRONOLACTON**)
# Việc xóa thẻ in đậm ở tiêu đề giúp Metadata gọn gàng và chính xác hơn khi truy xuất.
def clean_markdown_headers(text):
    # Xóa các dấu ** bao quanh text nằm ngay sau các dấu #
    return re.sub(r'(#+)\s*\*\*(.*?)\*\*', r'\1 \2', text)

In [12]:
# Khai báo các cấp độ Header muốn tách và tên Metadata tương ứng
headers_to_split_on = [
    ("#", "Heading_I"),          # Tương ứng với # SPIRONOLACTON
    ("##", "Heading_II"),         # Tương ứng với ## Dược lý và cơ chế tác dụng
    ("###", "Heading_III"),
]

In [13]:
# Khởi tạo Markdown Splitter
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=True # True: Xóa luôn dấu # trong text chunk; False: Giữ lại
)


In [14]:
MAX_TOKENS = 8191 - 50

In [15]:
# Splitter 2: Cắt các đoạn quá dài theo đúng số lượng TOKEN thay vì ký tự
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="text-embedding-3-large", # Đổi tên model tương ứng nếu cần (vd: gpt-4o)
    chunk_size=MAX_TOKENS,      # Giới hạn 250 TOKENS
    chunk_overlap=0,            # Chồng lấp 0 TOKENS
)

In [16]:
def process_document(file_path):
    raw_text = read_markdown_file(file_path)
    cleaned_text = clean_markdown_headers(raw_text)
    
    # Bước 1: Cắt theo Header và gắn Metadata
    md_header_splits = markdown_splitter.split_text(cleaned_text)
    
    # Bước 2: Cắt nhỏ tiếp các đoạn quá dài (Metadata vẫn được bảo toàn)
    final_splits = text_splitter.split_documents(md_header_splits)
    
    return final_splits

In [17]:
# Chạy thử nghiệm
file_path = r"Data/Huong_Dan_Su_Dung/page_39_97.md" 
chunks = process_document(file_path)

# In thử 2 chunk đầu tiên để kiểm tra
for i, chunk in enumerate(chunks[:2]):
    print(f"--- Chunk {i+1} ---")
    print(f"Metadata: {chunk.metadata}")
    print(f"Content: {chunk.page_content}\n")

--- Chunk 1 ---
Metadata: {'Heading_I': 'HƯỚNG DẪN SỬ DỤNG DƯỢC THƯ QUỐC GIA VIỆT NAM'}
Content: Nhiều nước trên thế giới xuất bản Dược thư quốc gia. Một số nước chỉ soạn thảo Dược thư quốc gia ngắn gọn loại bỏ túi để giúp thầy thuốc tra cứu khi làm việc. Dược thư quốc gia Việt Nam là sách hướng dẫn sử dụng thuốc an toàn, hợp lý và hiệu quả do Bộ Y tế ban hành. Dược thư quốc gia Việt Nam lần xuất bản thứ nhất được biên soạn trong khuôn khổ chương trình hợp tác y tế Việt Nam - Thụy Điển theo một quy trình chặt chẽ để cung cấp cho các bác sỹ, dược sỹ và cán bộ y tế các thông tin về thuốc, nhằm hướng tới sử dụng thuốc hợp lý, an toàn và hiệu quả.  
Từ năm 2011 - 2013, Bộ Y tế đã tổ chức biên soạn cuốn Dược thư quốc gia Việt Nam lần xuất bản thứ hai với khoảng 700 dược chất trong số hơn 1 000 dược chất có trong hơn 10 000 dược phẩm lưu hành trên thị trường Việt Nam, bao gồm các thuốc có trong Danh mục thuốc thiết yếu Việt Nam, Danh mục các thuốc tân dược thuộc phạm vi thanh toán của quỹ bả

In [18]:
import pandas as pd

# 1. Chuyển đổi danh sách chunks thành danh sách các dictionary
data_to_save = []
for chunk in chunks:
    # Gom nội dung text và phân tách metadata thành các key-value
    row = {
        "Content": chunk.page_content,
        **chunk.metadata  # Giải nén các trường trong metadata (ví dụ: Ten_Thuoc, Muc_Chinh...)
    }
    data_to_save.append(row)

# 2. Tạo DataFrame từ danh sách trên
df = pd.DataFrame(data_to_save)

# 3. Lưu ra file CSV
csv_file_path = "Data/Huong_Dan_Su_Dung/page_39_97_chunk.csv"

# Sử dụng utf-8-sig để Excel có thể đọc tiếng Việt có dấu mà không bị lỗi font
df.to_csv(csv_file_path, index=False, encoding='utf-8-sig')

print(f"Đã lưu thành công {len(chunks)} chunks ra file: {csv_file_path}")

Đã lưu thành công 113 chunks ra file: Data/Huong_Dan_Su_Dung/page_39_97_chunk.csv
